В цьому домашньому завданні ми знову працюємо з даними з нашого змагання ["Bank Customer Churn Prediction (DLU Course)"](https://www.kaggle.com/t/7c080c5d8ec64364a93cf4e8f880b6a0).

Тут ми побудуємо рішення задачі класифікації з використанням kNearestNeighboors, знайдемо оптимальні гіперпараметри для цього методу і зробимо базові ансамблі. Це дасть змогу порівняти перформанс моделі з попередніми вивченими методами.

0. Зчитайте дані `train.csv` та зробіть препроцесинг використовуючи написаний Вами скрипт `process_bank_churn.py` так, аби в результаті отримати дані в розбитті X_train, train_targets, X_val, val_targets для експериментів.

  Якщо Вам не вдалось реалізувати в завданні `2.3. Дерева прийняття рішень` скрипт `process_bank_churn.py` - можна скористатись готовим скриптом з запропонованого рішення того завдання.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import sys
sys.path.append('/content/drive/MyDrive/Colab Notebooks')

In [16]:
import importlib
import process_bank_churn

importlib.reload(process_bank_churn)

from process_bank_churn import preprocess_data, preprocess_new_data

In [17]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score
import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
import numpy as np
import time
from sklearn.model_selection import RandomizedSearchCV

In [5]:
raw_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/train.csv')
data = preprocess_data(raw_df, scaler_numeric=True)

X_train, y_train = data['X_train'], data['train_targets']
X_val, y_val = data['X_val'], data['val_targets']

1. Навчіть на цих даних класифікатор kNN з параметрами за замовченням і виміряйте точність з допомогою AUROC на тренувальному та валідаційному наборах. Зробіть заключення про отриману модель: вона хороша/погана, чи є high bias/high variance?

In [6]:
knn_default = KNeighborsClassifier()
knn_default.fit(X_train, y_train)

train_probs = knn_default.predict_proba(X_train)[:, 1]
val_probs = knn_default.predict_proba(X_val)[:, 1]

train_auc = roc_auc_score(y_train, train_probs)
val_auc = roc_auc_score(y_val, val_probs)

print(f"kNN AUROC Train: {train_auc:.4f}")
print(f"kNN AUROC Val:   {val_auc:.4f}")
print(f"Різниця: {train_auc - val_auc:.4f}")

kNN AUROC Train: 0.9610
kNN AUROC Val:   0.8750
Різниця: 0.0860


Різниця 0.086 вказує на те, що модель має high variance

2. Використовуючи `GridSearchCV` знайдіть оптимальне значення параметра `n_neighbors` для класифікатора `kNN`. Псотавте крос валідацію на 5 фолдів.

  Після успішного завершення пошуку оптимального гіперпараметра
    - виведіть найкраще значення параметра
    - збережіть в окрему змінну `knn_best` найкращу модель, знайдену з `GridSearchCV`
    - оцініть якість передбачень  `knn_best` на тренувальній і валідаційній вибірці з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи стала вона краще порівняно з попереднім пукнтом (2) цього завдання? Чи є вона краще за дерево прийняття рішень з попереднього ДЗ?

In [10]:
param_grid = {'n_neighbors': range(1, 31, 2)}

knn_search = GridSearchCV(
    KNeighborsClassifier(),
    param_grid,
    scoring='roc_auc',
    cv=5,
    n_jobs=-1
)

knn_search.fit(X_train, y_train)

print(f"Найкраще значення n_neighbors: {knn_search.best_params_['n_neighbors']}")
print(f"Найкращий результат AUROC на крос-валідації: {knn_search.best_score_:.4f}")

knn_best = knn_search.best_estimator_

train_probs_best = knn_best.predict_proba(X_train)[:, 1]
val_probs_best = knn_best.predict_proba(X_val)[:, 1]

train_auc_best = roc_auc_score(y_train, train_probs_best)
val_auc_best = roc_auc_score(y_val, val_probs_best)

print(f"kNN_best AUROC Train: {train_auc_best:.4f}")
print(f"kNN_best AUROC Val:   {val_auc_best:.4f}")
print(f"Різниця: {train_auc_best - val_auc_best:.4f}")

Найкраще значення n_neighbors: 29
Найкращий результат AUROC на крос-валідації: 0.9132
kNN_best AUROC Train: 0.9340
kNN_best AUROC Val:   0.9136
Різниця: 0.0203


Значення AUROC на валідації зросло з 0.875 до 0.9136. Модель тепер не має перенавчання і добре генералізує. kNN трохи програє дереву рішень.

3. Виконайте пошук оптимальних гіперпараметрів для `DecisionTreeClassifier` з `GridSearchCV` за сіткою параметрів
  - `max_depth` від 1 до 20 з кроком 2
  - `max_leaf_nodes` від 2 до 10 з кроком 1

  Обовʼязково при цьому ініціюйте модель з фіксацією `random_state`.

  Поставте кросвалідацію на 3 фолди, `scoring='roc_auc'`, та виміряйте, скільки часу потребує пошук оптимальних гіперпараметрів.

  Після успішного завершення пошуку оптимальних гіперпараметрів
    - виведіть найкращі значення параметра
    - збережіть в окрему змінну `dt_best` найкращу модель, знайдену з `GridSearchCV`
    - оцініть якість передбачень  `dt_best` на тренувальній і валідаційній вибірці з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи ця модель краща за ту, що ви знайшли вручну?

In [14]:
tree_clf = DecisionTreeClassifier(random_state=42)

param_grid_tree = {
    'max_depth': range(1, 21, 2),
    'max_leaf_nodes': range(2, 11)
}

tree_search = GridSearchCV(
    tree_clf,
    param_grid_tree,
    scoring='roc_auc',
    cv=3,
    n_jobs=-1
)

start_time = time.time()
tree_search.fit(X_train, y_train)
execution_time = time.time() - start_time

print(f"Пошук завершено за {execution_time:.2f} секунд")
print(f"Найкращі параметри дерева: {tree_search.best_params_}")
print(f"Найкращий AUROC: {tree_search.best_score_:.4f}")

tree_best = tree_search.best_estimator_

train_auc_tree = roc_auc_score(y_train, tree_best.predict_proba(X_train)[:, 1])
val_auc_tree = roc_auc_score(y_val, tree_best.predict_proba(X_val)[:, 1])
print(f"AUROC на тренуванні: {train_auc_tree:.4f}")
print(f"AUROC на валідації: {val_auc_tree:.4f}")
print(f"Різниця: {train_auc_tree - val_auc_tree:.4f}")

Пошук завершено за 7.40 секунд
Найкращі параметри дерева: {'max_depth': 5, 'max_leaf_nodes': 10}
Найкращий AUROC: 0.9014
AUROC на тренуванні: 0.9015
AUROC на валідації: 0.9002
Різниця: 0.0013


З точки зору точності ручна модель показує кращий результат 0.9241. Параметри ручного підбору лежать поза межами сітки, тому GridSearchCV до нього не дойшов.

4. Виконайте пошук оптимальних гіперпараметрів для `DecisionTreeClassifier` з `RandomizedSearchCV` за заданою сіткою параметрів і кількість ітерацій 40.

  Поставте кросвалідацію на 3 фолди, `scoring='roc_auc'`, зафіксуйте `random_seed` процедури крос валідації та виміряйте, скільки часу потребує пошук оптимальних гіперпараметрів.

  Після успішного завершення пошуку оптимальних гіперпараметрів
    - виведіть найкращі значення параметра
    - збережіть в окрему змінну `dt_random_search_best` найкращу модель, знайдену з `RandomizedSearchCV`
    - оцініть якість передбачень  `dt_random_search_best` на тренувальній і валідаційній вибірці з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи ця модель краща за ту, що ви знайшли з `GridSearch`?
    - проаналізуйте параметри `dt_random_search_best` і порівняйте з параметрами `dt_best` - яку бачите відмінність? Ця вправа потрібна аби зрозуміти, як різні налаштування `DecisionTreeClassifier` впливають на якість моделі.

In [15]:
params_dt = {
    'criterion': ['gini', 'entropy'],
    'splitter': ['best', 'random'],
    'max_depth': np.arange(1, 20),
    'max_leaf_nodes': np.arange(2, 20),
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': [None, 'sqrt', 'log2']
}

tree_random_search = RandomizedSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_distributions=params_dt,
    n_iter=40,
    scoring='roc_auc',
    cv=3,
    random_state=42,
    n_jobs=-1
)

start_time = time.time()
tree_random_search.fit(X_train, y_train)
execution_time = time.time() - start_time

print(f"Пошук завершено за {execution_time:.2f} секунд")
print(f"Найкращі параметри: {tree_random_search.best_params_}")
print(f"Найкращий AUROC на крос-валідації: {tree_random_search.best_score_:.4f}")

best_tree = tree_random_search.best_estimator_

train_auc = roc_auc_score(y_train, best_tree.predict_proba(X_train)[:, 1])
val_auc = roc_auc_score(y_val, best_tree.predict_proba(X_val)[:, 1])

print(f"AUROC на тренуванні: {train_auc:.4f}")
print(f"AUROC на валідації: {val_auc:.4f}")
print(f"Різниця: {train_auc - val_auc:.4f}")

Пошук завершено за 1.34 секунд
Найкращі параметри: {'splitter': 'best', 'min_samples_split': 20, 'min_samples_leaf': 2, 'max_leaf_nodes': np.int64(14), 'max_features': None, 'max_depth': np.int64(16), 'criterion': 'entropy'}
Найкращий AUROC на крос-валідації: 0.9109
AUROC на тренуванні: 0.9169
AUROC на валідації: 0.9166
Різниця: 0.0003


Різниця між тренувальним та валідаційним AUROC - 0.0003, добра стабільність. Ця модель краща за GridSearch. RandomizedSearchCV показує кращі результати оскільки шукає в ширшому діапазоні параметрів. Хоча ручний підбір параметрів все ще має вищий AUROC

5. Якщо у Вас вийшла метрика `AUROC` в цій серії експериментів - зробіть ще один `submission` на Kaggle і додайте код для цього і скріншот скора на публічному лідерборді нижче.

  Сподіваюсь на цьому етапі ви вже відчули себе справжнім дослідником 😉

In [18]:
test_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/test.csv')

X_test_processed = preprocess_new_data(
    test_df,
    input_cols=data['input_cols'],
    scaler=data['scaler'],
    encoder=data['encoder']
)

test_probs = best_tree.predict_proba(X_test_processed)[:, 1]

submission_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/sample_submission.csv')

submission_df['Exited'] = test_probs
submission_df.to_csv('/content/drive/MyDrive/Colab Notebooks/submission_random_search_tree.csv', index=False)
print(submission_df.head())

      id    Exited
0  15000  0.237911
1  15001  0.012115
2  15002  0.203947
3  15003  0.569848
4  15004  0.082171
